In [1]:
import sys
sys.path.insert(0, "/scratch/kanniain/pst/MarS")  # folder that contains market_simulation/
from market_simulation.models.order_model import OrderModel


2026-01-07 11:43:36,520 - /scratch/kanniain/pst/MarS/market_simulation/__init__.py:15 - INFO - init logging


/PUHTI_TYKKY_Quvj2Tb/miniforge/envs/env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
import pandas as pd
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader
from market_simulation.models.order_model import OrderModel



# -------------------------
# Config
# -------------------------
K = 1024
VAL_FRACTION = 0.10          # last 10% of time = validation
BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-4

# ~2M params
EMB_DIM = 32
NUM_LAYERS = 2
NUM_HEADS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Load features
# -------------------------
features_df = pd.read_parquet("../data/mymessages.parquet")

print(features_df)
breakpoint()

t_sec = (features_df["Time"].to_numpy() // 1_000_000_000).astype("int64")  # ns -> seconds
features_df["f4"] = (t_sec - 34200).clip(0, 23399)  # seconds since 09:30

F_mat = features_df[[f"f{i}" for i in range(15)]].to_numpy(dtype=np.int64)  # (T, 15)
T = len(F_mat)
split = int(T * (1.0 - VAL_FRACTION))

class WindowDataset(Dataset):
    """Return length-K windows fully inside [start, end)."""
    def __init__(self, mat: np.ndarray, K: int, start: int, end: int):
        self.stride = 8
        self.mat = mat
        self.K = K
        self.start = start
        self.end = end
        self.n = max(0, ((end - start) - K) // self.stride + 1) #max(0, (end - start) - K + 1)

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, i: int) -> torch.Tensor:
        #idx = self.start + i
        idx = self.start + i * self.stride
        return torch.from_numpy(self.mat[idx: idx + self.K]).long()  # (K, 15)

train_end = max(split, K)
val_start = min(split, T - K)

train_ds = WindowDataset(F_mat, K, start=0, end=train_end)
val_ds   = WindowDataset(F_mat, K, start=val_start, end=T)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total rows: {T:,}")
print(f"Train windows: {len(train_ds):,} | Val windows: {len(val_ds):,} (last {VAL_FRACTION:.0%} of time)")

# -------------------------
# Model
# -------------------------
model = OrderModel(
    emb_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    num_max_orders=K,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.2f}M")

opt = torch.optim.AdamW(model.parameters(), lr=LR)

def lm_loss(logits: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    targets = X[:, :, 0]             # (B, K)
    logits_s  = logits[:, :-1, :]    # (B, K-1, vocab)
    targets_s = targets[:, 1:]       # (B, K-1)
    return F.cross_entropy(
        logits_s.reshape(-1, logits_s.size(-1)),
        targets_s.reshape(-1),
    )

@torch.no_grad()
def eval_val_loss(epoch: int) -> float:
    model.eval()
    total = 0.0
    count = 0
    pbar = tqdm(val_dl, desc=f"val  epoch {epoch:02d}", leave=False)
    for X in pbar:
        X = X.to(device, non_blocking=True)
        logits = model(X)
        loss = lm_loss(logits, X)

        n = X.size(0) * (X.size(1) - 1)   # predicted positions
        total += loss.item() * n
        count += n

        pbar.set_postfix(loss=f"{(total/max(1,count)):.4f}")
    return total / max(1, count)

# -------------------------
# Train + print VAL loss
# -------------------------
for epoch in range(1, EPOCHS + 1):
    model.train()
    pbar = tqdm(train_dl, desc=f"train epoch {epoch:02d}", leave=True)

    for X in pbar:
        X = X.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        logits = model(X)
        loss = lm_loss(logits, X)
        loss.backward()
        opt.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss = eval_val_loss(epoch)
    print(f"epoch {epoch:02d}  val_loss {val_loss:.4f}")


               i            Time     f0  f1  f2  f3     f4  f5  f6  f7  f8  \
0          24903  34200010331091  38377   9   9  50  19799   0   0   0   0   
1          24904  34200010332878  43488   9   9  50  19799   0   0   0   0   
2          24905  34200010409495  13696   9   0  50  19799   0   0   0   0   
3          24906  34200010411684  19328   9   0  50  19799   0   0   0   0   
4          24907  34200020823998  21989   9   0  50  19799   0   0   0   0   
...          ...             ...    ...  ..  ..  ..    ...  ..  ..  ..  ..   
2312300  2363140  57599993081842   9104   5   0 -98  43199   1  32   0  32   
2312301  2363141  57599997780640   9716   7   0 -98  43199   1  32   0  32   
2312302  2363144  57600000277067  39300   0   9 -98  43199   1  32   0  32   
2312303  2363145  57600000277067  39332   0   9 -98  43199   1  32   0  32   
2312304  2363146  57600000277067  39412   3   9 -98  43199   1  32   0  32   

         f9  f10  f11  f12  f13  f14  
0         0    0    0   

train epoch 01:   7%|▋         | 557/8126 [01:58<26:51,  4.70it/s, loss=5.6725]